# Salud Chilecito - Demo DAO y plataforma

Este notebook funciona como recorrido de presentacion de Salud Chilecito. Primero se valida el modelo de datos de demo, despues se revisa el DAO y finalmente se explican los comandos para conectar con Oracle.

**Formas de uso del proyecto:**

- `http://localhost:8000`: plataforma grafica.
- `http://localhost:8000/bot`: plataforma conversacional con Bot IA local.
- `src/dao`: capa DAO para Oracle.
- `sql/`: scripts Oracle con tablespaces, usuarios, esquema, indices, seed y validaciones.


## 1. Preparacion

Ejecutar el notebook desde la raiz del repositorio:

```bash
Windows: .\scripts\windows\04_abrir_notebook.ps1
Ubuntu: bash scripts/ubuntu/04_abrir_notebook.sh
```

Si todavia no se instalaron dependencias:

```bash
python -m venv .venv
.venv\Scripts\activate        # Windows
source .venv/bin/activate      # Ubuntu
pip install -r requirements.txt
```


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

print("Repositorio:", ROOT)
print("Existe src/dao:", (ROOT / "src" / "dao").exists())
print("Existe sql/:", (ROOT / "sql").exists())


## 2. Datos de demo

La plataforma web puede funcionar aunque Oracle no este levantado. Para eso usa `data/demo_seed.json` y guarda cambios de prueba en `runtime/salud_chilecito_data.json`.


In [ ]:
seed_path = ROOT / "data" / "demo_seed.json"
data = json.loads(seed_path.read_text(encoding="utf-8"))

for name in ["centros", "especialidades", "medicos", "pacientes", "turnos", "agendas", "tarifas", "documentos"]:
    print(f"{name}: {len(data[name])}")


In [ ]:
import pandas as pd

pd.DataFrame(data["centros"])


## 3. Uso del store local

Esta parte demuestra operaciones reales sin tocar Oracle. Sirve para probar la interfaz y validar la funcionalidad sin preparar una base externa.


In [ ]:
from tempfile import TemporaryDirectory
from src.webapp.store import JsonStore

tmp = TemporaryDirectory()
demo_store = JsonStore(
    runtime_path=Path(tmp.name) / "runtime.json",
    seed_path=seed_path,
    uploads_dir=Path(tmp.name) / "uploads",
)

paciente = demo_store.create_patient({
    "dni": "50999888",
    "nombre": "Paciente Notebook",
    "telefono": "3825-111111",
    "distrito": "Chilecito",
    "obra_social": "APOS",
})

turno = demo_store.create_turno({
    "paciente_id": paciente["id"],
    "medico_id": 1,
    "fecha": "2026-06-20",
    "hora": "09:30",
    "estado": "CONFIRMADO",
    "precio": 0,
    "motivo": "Control desde notebook",
})

print("Paciente creado:", paciente)
print("Turno creado:", turno["id"], turno["paciente"]["nombre"], turno["medico"]["nombre"])


## 4. Bot IA local

El bot no llama a servicios externos: interpreta comandos frecuentes y opera el mismo store de la plataforma. Esto permite demostrar una segunda forma de uso, conversacional, sin depender de una API paga.


In [ ]:
from src.webapp.bot_agent import BotAgent

bot = BotAgent(demo_store)
respuesta = bot.handle("listar turnos")
print(respuesta["reply"])


## 5. Capa DAO para Oracle

Los DAO son la interfaz formal entre Python y Oracle. La aplicacion no deberia armar consultas dispersas por todos lados; las consultas quedan agrupadas en clases como `CentroDAO`, `PacienteDAO` y `TurnoDAO`.


In [ ]:
from src.dao import CentroDAO, PacienteDAO, TurnoDAO

print("DAO disponibles:")
print("-", CentroDAO.__name__)
print("-", PacienteDAO.__name__)
print("-", TurnoDAO.__name__)


In [ ]:
## 5. Capa DAO para Oracle

Los DAO son la interfaz formal entre Python y Oracle. La aplicacion no deberia armar consultas dispersas por todos lados; las consultas quedan agrupadas en clases como `CentroDAO`, `PacienteDAO` y `TurnoDAO.

**Nuevos métodos del modelo Single-Hospital:**
- `listar_sintomas()` - Lista síntomas con especialidades
- `buscar_especialidad_por_sintoma(sintoma)` - Busca especialidad por síntoma
- `obtener_configuracion_hospital(id)` - Configuración del hospital
- `listar_tipos_consulta()` - Tipos de consulta disponibles
- `obtener_precios_por_especialidad(centro, especialidad)` - Precios por especialidad

In [ ]:
## 5.1 Ejemplos del DAO con nuevas funcionalidades (Modelo Single-Hospital)

### Selección por síntomas
# El paciente selecciona sus síntomas y el sistema sugiere la especialidad adecuada
# Esta funcionalidad mejora la experiencia de usuario al derivar automáticamente

# Ejemplo de uso del DAO con síntomas
# from dao import SaludDAO
# dao = SaludDAO()
# 
# # Buscar especialidad por síntoma
# resultado = dao.buscar_especialidad_por_sintoma("dolor de pecho")
# if resultado:
#     print(f"Para 'dolor de pecho' se recomienda: {resultado['especialidad']}")
#     print(f"Prioridad: {resultado['prioridad']}")
# 
# # Listar todos los síntomas disponibles
# sintomas = dao.listar_sintomas()
# print(f"\nSíntomas disponibles: {len(sintomas)}")
# for s in sintomas:
#     print(f"  - {s['descripcion']} → {s['especialidad']} (Prioridad: {s['prioridad']})")

### Precios por tipo de consulta
# Cada especialidad tiene rangos de precios según el tipo de consulta
# Esto permite que cada hospital defina su propia política de precios

# precios = dao.obtener_precios_por_especialidad(centro_id=1, especialidad_id=3)
# print("\nPrecios por tipo de consulta:")
# for precio in precios:
#     print(f"  {precio['tipo_consulta']['nombre']}: ${precio['precio_estimado']}")
#     print(f"    Rango: ${precio['precio_minimo']} - ${precio['precio_maximo']}")
#     print(f"    Duración: {precio['duracion_minutos']} minutos")

### Configuración del hospital
# Cada instancia puede personalizar su configuración (modelo single-hospital)
# Esto permite que cada hospital tenga su propia identidad visual y políticas

# config = dao.obtener_configuracion_hospital(id=1)
# if config:
#     print(f"\nConfiguración del Hospital:")
#     print(f"  Nombre: {config['nombre_hospital']}")
#     print(f"  Centro Principal: {config['centro_principal']}")
#     print(f"  Mensaje de Bienvenida: {config['mensaje_bienvenida']}")
#     print(f"  Color Primario: {config['color_primario']}")
#     print(f"  Color Secundario: {config['color_secundario']}")
#     print(f"  Tiempo de Cancelación: {config['tiempo_cancelacion_horas']} horas")

### Disponibilidad mejorada
# Visualización de horarios disponibles por médico y fecha específica
# Esta funcionalidad muestra días y horarios específicos para los próximos 7 días

# disponibilidad = dao.obtener_turnos_disponibles_por_medico(medico_id=1, dias=7)
# print(f"\nDisponibilidad para los próximos 7 días:")
# for slot in disponibilidad:
#     print(f"  Fecha: {slot['fecha']}")
#     print(f"  Día: {slot['dia_semana']}")
#     print(f"  Horario: {slot['hora_inicio']} - {slot['hora_fin']}")
#     print(f"  Cupos disponibles: {slot['cupos_disponibles']}/{slot['cupo_diario']}")
#     print(f"  Duración: {slot['duracion_minutos']} minutos")
#     print()

# Horarios específicos para una fecha
# horarios = dao.obtener_horarios_disponibles(medico_id=1, fecha="2026-06-20")
# print(f"Horarios disponibles para el 2026-06-20:")
# for h in horarios:
#     print(f"  {h['hora']} - {h['hora_fin']}")

class FakeDB:
    def __init__(self):
        self.calls = []

    def fetch_all(self, sql, params=None):
        self.calls.append(("fetch_all", " ".join(sql.split()), params or {}))
        return []

    def fetch_one(self, sql, params=None):
        self.calls.append(("fetch_one", " ".join(sql.split()), params or {}))
        return {"total": 0}

    def execute(self, sql, params=None):
        self.calls.append(("execute", " ".join(sql.split()), params or {}))
        return 1

fake_db = FakeDB()
CentroDAO(fake_db).listar(distrito="Chilecito")
fake_db.calls[-1]

## 6. Sistema de Integración con HIS (Hospital Information Systems)

**Problema**: Los hospitales ya tienen sistemas de gestión (HIS) y no quieren reemplazarlos.

**Solución**: Salud Chilecito se integra con los sistemas existentes mediante API REST, no los reemplaza.

### Características del sistema de integración

- **API Keys**: Autenticación segura para integraciones entre sistemas
- **Webhooks**: Sincronización bidireccional en tiempo real
- **Adaptadores**: Soporte para diferentes tipos de HIS (REST, FHIR)
- **Logs de auditoría**: Registro de todas las operaciones de integración

### Ejemplo de uso de API Keys

```python
from src.webapp.auth import api_key_manager

# Generar una API Key para un hospital
api_key = api_key_manager.generate_key(
    hospital_name="Hospital Eleazar Herrera Motta",
    hospital_id=1,
    permissions=["read", "write"]
)
print(f"API Key generada: {api_key}")

# Validar la API Key
hospital_info = api_key_manager.validate_key(api_key)
print(f"Información del hospital: {hospital_info}")
```

### Ejemplo de uso de Webhooks

```python
from src.webapp.webhooks import webhook_manager, EventTypes

# Registrar un webhook para recibir notificaciones
webhook_id = webhook_manager.register_webhook(
    hospital_id=1,
    url="https://hospital-ejemplo.com/webhooks/salud-chilecito",
    events=[EventTypes.TURNO_CREATED, EventTypes.TURNO_CANCELLED],
    secret="secreto-para-firmar-payloads"
)
print(f"Webhook registrado: {webhook_id}")

# Disparar un evento de prueba
webhook_manager.trigger_event(
    EventTypes.TURNO_CREATED,
    {
        "id": 1,
        "paciente_id": 1,
        "medico_id": 1,
        "fecha": "2026-06-20",
        "hora": "09:30"
    }
)
```

### Ejemplo de uso de Adaptadores

```python
from src.webapp.adapters import AdapterFactory, EXAMPLE_REST_CONFIG

# Crear un adaptador REST para un HIS
config = EXAMPLE_REST_CONFIG.copy()
config["base_url"] = "https://api.hospital-ejemplo.com"
config["api_key"] = "tu-api-key-aqui"

adapter = AdapterFactory.create_adapter("rest", config)

# Sincronizar pacientes con el HIS
pacientes = [
    {"dni": "12345678", "nombre": "Juan Pérez", "telefono": "3825-123456"}
]
synced = adapter.sync_patients(pacientes)
print(f"Pacientes sincronizados: {len(synced)}")
```

### Arquitectura de Integración

```
Paciente → Salud Chilecito → API REST → Adaptador → HIS (Sistema existente)
         ←                ←         ←         ←
```

Salud Chilecito actúa como una **capa de mejora** que se conecta a los sistemas existentes de los hospitales, agregando funcionalidades como:
- Selección por síntomas con IA
- Bot conversacional
- Landing pages personalizadas
- Visualización mejorada de disponibilidad

### Documentación

- [Arquitectura de Integración](../docs/ARQUITECTURA_INTEGRACION.md) - Guía completa de integración
- [Documentación de API](../docs/API_OPENAPI.md) - Referencia completa de la API REST
- [Ejemplos de Integración](../examples/integracion_his.py) - Ejemplos de código completos

## 7. Resumen para el Profesor

**Modelo de negocio**: Sistema vendido a hospitales/clínicas (single-hospital instance)

**Autores**: Alesandro David Fajardo / Kevin Facundo Nunez  
**Universidad**: Universidad Nacional de Chilecito  
**Año**: 2026

### Características principales de la plataforma

1. **Gestión completa**: Centros, médicos, pacientes, turnos, agendas, documentos
2. **Selección por síntomas**: Derivación automática a especialidades (dolor de pecho → Cardiología)
3. **Precios personalizados**: Por especialidad y tipo de consulta (General, Urgencia, Seguimiento)
4. **Configuración por hospital**: Branding personalizado (logo, colores, mensajes de bienvenida)
5. **Disponibilidad en tiempo real**: Días y horarios específicos para los próximos 7 días
6. **Integración con HIS**: Se conecta a sistemas existentes, no los reemplaza
7. **Bot IA local**: Interfaz conversacional sin dependencias externas
8. **API REST estándar**: Para integración con otros sistemas

### Arquitectura técnica

- **Base de datos**: Oracle con scripts completos (tablespaces, usuarios, esquema, índices, seed)
- **Capa DAO**: Python con patrón Data Access Object para abstracción de base de datos
- **API REST**: Endpoints estándar con documentación OpenAPI
- **Frontend**: JavaScript con visualización mejorada de disponibilidad
- **Bot IA**: Local sin dependencias de servicios externos

### Ventajas competitivas

1. **No reemplaza**: Los hospitales mantienen sus sistemas existentes
2. **Fácil integración**: API estándar y documentación clara
3. **Valor agregado**: Funcionalidades que los HIS no tienen (IA, selección por síntomas)
4. **Flexibilidad**: Se adapta a diferentes sistemas mediante adaptadores
5. **Costo efectivo**: Menor costo que migrar a un sistema completo

### Comandos para ejecutar

```bash
# Instalar dependencias
python -m venv .venv
.venv\Scripts\activate        # Windows
source .venv/bin/activate      # Ubuntu
pip install -r requirements.txt

# Ejecutar servidor
python -m src.webapp.server
```

### URLs de acceso

- `http://localhost:8000` - Plataforma gráfica principal
- `http://localhost:8000/bot` - Bot IA conversacional
- `http://localhost:8000/landing` - Landing page del hospital

### Documentación completa

- [README](../README.md) - Documentación general del proyecto
- [Arquitectura de Integración](../docs/ARQUITECTURA_INTEGRACION.md) - Guía de integración con HIS
- [Documentación de API](../docs/API_OPENAPI.md) - Referencia completa de la API REST
- [Ejemplos de Integración](../examples/integracion_his.py) - Ejemplos de código completos
- [Integración con Hospital](../docs/INTEGRACION_HOSPITAL.md) - Guía de integración específica

## 7. Comandos finales de presentacion

```bash
python scripts/check_requirements.py
python -m pytest -q
python -m src.webapp.server
```

Abrir:

- `http://localhost:8000`
- `http://localhost:8000/bot`
